# Stage 2: Exploratory Data Analysis (EDA) Notebook
## Amazon ML Challenge — Business Entity Resolution

This notebook analyzes linguistic properties of business names (legal suffixes, case variations, punctuation, token frequencies), address structures (numbers, postal/PIN codes, abbreviations), country open-set patterns, and ground-truth match cardinalities.

In [ ]:
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.stage_runner import StageController
from src.utils.storage import StorageManager
from src.data.eda import run_eda

### Step 1: Execute EDA Stage

In [ ]:
storage = StorageManager("../configs/config.yaml")
storage.initialize_directories()

# Run EDA stage
controller = StageController("../configs/config.yaml")
success = controller.run_stage("eda", force=True)

### Step 2: Display Summary Findings

In [ ]:
eda_path = storage.artifacts_dir / "eda_summary.json"
if eda_path.exists():
    with open(eda_path, "r", encoding="utf-8") as f:
        eda_data = json.load(f)
    
    print("=" * 80)
    print("                 STAGE 2: EDA SUMMARY FINDINGS                 ")
    print("=" * 80)
    
    bn = eda_data.get("business_name_analysis", {})
    if bn:
        print("\n--- BUSINESS NAME ANALYSIS ---")
        suf = bn.get("legal_suffixes", {})
        print(f"  * Records with Legal Suffixes : {suf.get('names_with_legal_suffix_pct') }%")
        print(f"  * Top Legal Suffixes          : {suf.get('top_legal_suffixes')}")
        print(f"  * Casing Distribution         : {bn.get('casing_and_punctuation', {}).get('casing_distribution')}")
        print(f"  * Frequent Name Tokens        : {bn.get('frequent_name_tokens')}")
    
    addr = eda_data.get("address_analysis", {})
    if addr:
        print("\n--- BUSINESS ADDRESS ANALYSIS ---")
        struct = addr.get("structural_patterns", {})
        print(f"  * Addresses with Numbers      : {struct.get('addresses_with_numbers_pct')}%")
        print(f"  * US 5-Digit Zip Matches      : {struct.get('addresses_with_5digit_us_postal_pct')}%")
        print(f"  * India 6-Digit PIN Matches   : {struct.get('addresses_with_6digit_india_pin_pct')}%")
        print(f"  * Top Abbreviations           : {struct.get('top_address_abbreviations')}")
    
    print("\n--- KEY IMPLICATIONS FOR DOWNSTREAM STAGES ---")
    for imp in eda_data.get("key_findings_and_implications", []):
        print(f"  -> {imp}")
    print("=" * 80)
else:
    print("EDA summary not found. Please run Step 1.")